In [1]:
# -*- coding: utf-8 -*-
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: light
#       format_version: '1.5'
#       jupytext_version: 1.14.5
#   kernelspec:
#     display_name: Python 3 (ipykernel)
#     language: python
#     name: python3
# ---

# # Gerenciador de Experimentos de Otimização de L0 com AG

# Este notebook lê uma configuração de experimentos de um arquivo JSON
# e executa (ou resume) cada otimização.

# ## 1. Configurações e Imports Globais (Fixos)

# +
import pandas as pd
import numpy as np
import time
import os
import json
import glob 
import math
# import matplotlib.pyplot as plt # Comentado, pois o foco é a execução
# import seaborn as sns # Comentado

# --- Imports da Biblioteca ---
import sys
module_path = os.path.abspath(os.path.join('..')) # Assumindo que o notebook está em 'examples'
if module_path not in sys.path:
    sys.path.append(module_path)

try:
    from activetextclassification.data_preparation import load_split_and_preprocess_data
    from activetextclassification.models import get_model
    from activetextclassification.embeddings import get_embedder 
    from activetextclassification.optimization.genetic_l0_optimizer import GeneticL0Optimizer
    from activetextclassification.utils import preprocess_label
    print("Módulos da biblioteca activetextclassification importados.")
except Exception as e: 
    print(f"ERRO CRÍTICO ao importar biblioteca: {e}") 
    raise e

# --- Autoreload (Opcional, para desenvolvimento) ---
# try: 
#     %load_ext autoreload
#     %autoreload 2
#     print("Autoreload ativado.")
# except Exception: pass

# --- Parâmetros FIXOS do Dataset e Classificador ---
DATA_FILE_PATH = r'../data/dataset.csv' 
TEXT_COLUMN_NAME = 'nm_item'
LABEL_COLUMN_NAME = 'nm_product'
MIN_SAMPLES_PER_CLASS_PREPROC = 5
RARE_LABEL_GROUP = "_RARE_"
CLASSIFIER_SETUP_AG = { 
    'type': 'PVBin',
    'params': {'method':'binary', 'query':'binary', 'norm':None, 'query_norm':None, 'ngram_range': [1,2]}
}
GLOBAL_EMBEDDER_SETUP_AG = None 

# --- Parâmetros FIXOS para a Divisão de Dados ---
TEST_SET_FRACTION = 0.30
RANDOM_SEED_SPLIT = 42 

# --- Parâmetros GLOBAIS para os Experimentos AG ---
# Definidos aqui, mas podem ser sobrescritos por valores no JSON se presentes
# e a lógica de leitura do JSON for ajustada para isso.
# Conforme a nova orientação, estes são fixos:
GLOBAL_POPULATION_SIZE_AG = 50
GLOBAL_N_GENERATIONS_AG = 100

# Parâmetros do AG que mantêm seus defaults ou podem ser lidos do JSON se especificados lá
DEFAULT_CROSSOVER_RATE_AG = 0.8
DEFAULT_MUTATION_RATE_AG = 0.1 
DEFAULT_ELITISM_RATE_AG = 0.1
DEFAULT_TOURNAMENT_SIZE_AG = 3
DEFAULT_LOG_DETAILED_FITNESS = True

# --- Controle de Execução ---
CONFIG_FILE_PATH = 'experiments_config.json' 
FORCE_RESTART_ALL_OPTIMIZATIONS = False 
FORCE_DATA_REDIVISION = False       

print("Configurações globais fixas definidas.")
# -


# ## 2. Função para Executar uma Única Otimização (com Checkpointing)
# (MODIFICADA A ORDEM DOS PARÂMETROS)

# +
def execute_single_ag_optimization(
    df_l0_sampling_pool,         
    df_evaluation_set,           
    all_possible_labels_list,    
    global_embedder_instance,   
    
    # Parâmetros específicos do experimento atual (vindos do JSON ou calculados)
    l0_size_optim,
    population_size_to_use,
    n_generations_to_use,
    mutation_strength_to_use, 
    
    # Parâmetros para cada um dos 4 cenários (Acc Max/Min, F1 Max/Min) - SEM DEFAULT AQUI
    optimization_goal_scenario,    
    fitness_metric_name_scenario,  
    
    # Parâmetros de diretório e arquivo - SEM DEFAULT AQUI
    base_output_dir_experiment, 
    
    # Parâmetros com defaults globais (podem ser sobrescritos pelo JSON se fornecidos)
    # Estes DEVEM VIR DEPOIS dos argumentos sem default
    crossover_rate_ag = DEFAULT_CROSSOVER_RATE_AG, # DEFAULT_CROSSOVER_RATE_AG definido na Célula 1
    mutation_rate_ag = DEFAULT_MUTATION_RATE_AG,   # DEFAULT_MUTATION_RATE_AG definido na Célula 1
    elitism_rate_ag = DEFAULT_ELITISM_RATE_AG,     # DEFAULT_ELITISM_RATE_AG definido na Célula 1
    tournament_size_ag = DEFAULT_TOURNAMENT_SIZE_AG, # DEFAULT_TOURNAMENT_SIZE_AG definido na Célula 1
    log_detailed_fitness_ag = DEFAULT_LOG_DETAILED_FITNESS, # DEFAULT_LOG_DETAILED_FITNESS definido na Célula 1
    
    seed_offset_val = 0, # Mantido com default
    force_restart_this_ag_run = False # Mantido com default
    ):

    """
    Executa ou resume uma única instância de otimização do AG para um cenário.
    """
    os.makedirs(base_output_dir_experiment, exist_ok=True)
    
    goal_suffix = f"_{fitness_metric_name_scenario.split('_')[0].upper()}_{optimization_goal_scenario.upper()}"
    
    # Nomes para arquivos de RESULTADO FINAL
    final_history_file = os.path.join(base_output_dir_experiment, f"ag_history{goal_suffix}.xlsx")
    final_best_l0_file = os.path.join(base_output_dir_experiment, f"ag_best_l0{goal_suffix}.csv")
    detailed_log_path = os.path.join(base_output_dir_experiment, f"ag_detailed_fitness{fitness_metric_name_scenario.split('_')[0].upper()}_{optimization_goal_scenario.upper()}.csv")

    checkpoint_base_dir = os.path.join(base_output_dir_experiment, "checkpoints")
    checkpoint_file_name_prefix_for_ag = f"l0_{l0_size_optim}{goal_suffix}" 

    if not force_restart_this_ag_run and os.path.exists(final_history_file) and os.path.exists(final_best_l0_file):
        try:
            history_df_existing = pd.read_excel(final_history_file)
            generations_in_history = 0
            if not history_df_existing.empty and 'generation' in history_df_existing.columns:
                generations_in_history = history_df_existing['generation'].max()
            if generations_in_history >= n_generations_to_use:
                print(f"   Pulando {goal_suffix} para L0={l0_size_optim}: Resultados finais já existem com {generations_in_history} ger. (>= {n_generations_to_use} desejado).")
                return 
        except Exception as e_load:
            print(f"  AVISO: Não foi possível ler histórico existente para {goal_suffix}, L0={l0_size_optim}. Tentando rodar/resumir. Erro: {e_load}")

    print(f"\n  --- Iniciando/Resumindo AG para L0={l0_size_optim}, Pop={population_size_to_use}, Métrica={fitness_metric_name_scenario}, Objetivo={optimization_goal_scenario}, Total Ger.={n_generations_to_use}, MutStr={mutation_strength_to_use} ---")

    ag_optimizer = GeneticL0Optimizer(
        df_full=df_l0_sampling_pool,
        df_evaluation_set=df_evaluation_set,
        text_column=TEXT_COLUMN_NAME, # Usar as constantes globais
        label_column=LABEL_COLUMN_NAME, # Usar as constantes globais
        classifier_config=CLASSIFIER_SETUP_AG, # Usar as constantes globais
        initial_l0_size=l0_size_optim,
        all_possible_labels=all_possible_labels_list,
        population_size=population_size_to_use, 
        n_generations=n_generations_to_use,     
        crossover_rate=crossover_rate_ag,
        mutation_rate=mutation_rate_ag,
        mutation_strength=mutation_strength_to_use, 
        elitism_rate=elitism_rate_ag,
        fitness_metric=fitness_metric_name_scenario,
        optimization_goal=optimization_goal_scenario,
        tournament_size=tournament_size_ag,
        random_seed=42 + seed_offset_val,
        embedder=global_embedder_instance,
        log_detailed_fitness=log_detailed_fitness_ag,
        checkpoint_dir=checkpoint_base_dir, 
        checkpoint_prefix=checkpoint_file_name_prefix_for_ag 
    )
    
    if force_restart_this_ag_run and os.path.exists(ag_optimizer.checkpoint_file): 
        try:
            os.remove(ag_optimizer.checkpoint_file)
            print(f"    Checkpoint {ag_optimizer.checkpoint_file} removido (force_restart_this_ag_run=True).")
        except Exception as e_del_ckpt:
            print(f"    AVISO: Não foi possível remover checkpoint {ag_optimizer.checkpoint_file}: {e_del_ckpt}")

    best_l0_indices, best_actual_perf, history_df = ag_optimizer.run_optimization(
        detailed_log_file_path_from_notebook=detailed_log_path
    )

    if history_df is not None and not history_df.empty:
        history_df.to_excel(final_history_file, index=False)
        print(f"    Histórico final salvo: {final_history_file}")
    if best_l0_indices is not None and len(best_l0_indices) > 0:
        best_l0_df = df_l0_sampling_pool.iloc[best_l0_indices][[TEXT_COLUMN_NAME, LABEL_COLUMN_NAME]].copy()
        best_l0_df['metric_value_on_eval_set'] = best_actual_perf
        best_l0_df['metric_type'] = fitness_metric_name_scenario
        best_l0_df['optimization_goal'] = optimization_goal_scenario
        best_l0_df.to_csv(final_best_l0_file, index=False, encoding='utf-8-sig')
        print(f"    Melhor L0 final salvo: {final_best_l0_file}")
    
    print(f"  --- Concluído AG para L0={l0_size_optim}, Métrica={fitness_metric_name_scenario}, Objetivo={optimization_goal_scenario} ---")
    return
# -

# ## 3. Preparação Inicial dos Dados (Divisão Treino/Teste)
# (Esta célula roda uma vez para criar/carregar os splits)

# +
print(">>> Etapa 3: Preparação Inicial dos Dados (Divisão Treino/Teste) <<<")
# Diretório para salvar os splits, pode ser um subdiretório de um diretório de resultados global
# ou um local fixo se os splits são usados por múltiplos conjuntos de experimentos.
# Para este exemplo, vamos colocar na raiz do projeto em uma pasta 'data_splits_cache'
DATA_SPLITS_CACHE_DIR = os.path.join("..", "data_splits_cache") # Um nível acima de 'examples'

df_train_pool_main, df_test_eval_main, all_possible_labels_main = load_split_and_preprocess_data(
    file_path=DATA_FILE_PATH,
    text_column=TEXT_COLUMN_NAME,
    label_column=LABEL_COLUMN_NAME,
    min_samples_per_class=MIN_SAMPLES_PER_CLASS_PREPROC,
    rare_group_label=RARE_LABEL_GROUP,
    test_set_size=TEST_SET_FRACTION,
    random_state_split=RANDOM_SEED_SPLIT,
    output_dir=DATA_SPLITS_CACHE_DIR, # Salva/carrega os CSVs aqui
    force_split=FORCE_DATA_REDIVISION # Usa o parâmetro global
)

if df_train_pool_main is None or df_test_eval_main is None or all_possible_labels_main is None:
    raise SystemExit("FALHA CRÍTICA: Não foi possível carregar ou dividir os dados. Abortando o notebook.")

print(f"\nPool de Treino/Otimização (D_train_opt) principal: {len(df_train_pool_main)} amostras")
print(f"Conjunto de Teste T principal: {len(df_test_eval_main)} amostras")
print(f"Total de labels únicos para F1-score: {len(all_possible_labels_main)}")

# Preparar Embedder Global (se necessário, treinado APENAS em df_train_pool_main)
embedder_instance_main_ag = None 
clf_type_main_ag = CLASSIFIER_SETUP_AG.get('type')
if clf_type_main_ag in ['GNB', 'LSVC', 'LR', 'SGD'] and GLOBAL_EMBEDDER_SETUP_AG:
    print("\n--- Preparando Embedder Global (treinado em D_train_opt principal) ---")
    embedder_instance_main_ag = get_embedder(GLOBAL_EMBEDDER_SETUP_AG)
    embedder_instance_main_ag.fit(df_train_pool_main[TEXT_COLUMN_NAME].tolist(), df_train_pool_main[LABEL_COLUMN_NAME].tolist())
elif clf_type_main_ag in ['GNB', 'LSVC', 'LR', 'SGD'] and not GLOBAL_EMBEDDER_SETUP_AG: 
    print("AVISO: Configuração de Embedder global ausente para classificador do AG baseado em features, mas não é erro crítico se não for usado.")
# -

# ## 4. Carregar Configurações dos Experimentos e Executar Loop
# (MODIFICADO para salvar os parâmetros do experimento)

# +
print("\n>>> Etapa 4: Carregar Configurações e Executar Otimizações AG <<<")

try:
    with open(CONFIG_FILE_PATH, 'r') as f:
        experiments_config_list = json.load(f)
    print(f"Configurações de experimento carregadas de: {CONFIG_FILE_PATH}")
except FileNotFoundError:
    print(f"ERRO: Arquivo de configuração '{CONFIG_FILE_PATH}' não encontrado. Crie o arquivo JSON com as configurações.")
    experiments_config_list = [] 
except json.JSONDecodeError as e:
    print(f"ERRO: Falha ao decodificar JSON em '{CONFIG_FILE_PATH}': {e}")
    experiments_config_list = []

if not isinstance(experiments_config_list, list):
    print("ERRO: O arquivo de configuração JSON deve conter uma lista de configurações de experimento.")
    experiments_config_list = []


# Loop principal sobre as configurações de experimento
for exp_idx, exp_config in enumerate(experiments_config_list):
    print(f"\n\n{'='*20} EXECUTANDO EXPERIMENTO {exp_idx + 1} / {len(experiments_config_list)} {'='*20}")

    l0_size_current = exp_config.get("L0_SIZE_TO_OPTIMIZE")
    if l0_size_current is None:
        print(f"AVISO: L0_SIZE_TO_OPTIMIZE não encontrado na config do experimento {exp_idx + 1}. Pulando.")
        continue

    pop_size_current = exp_config.get("POPULATION_SIZE_AG", GLOBAL_POPULATION_SIZE_AG) # Pega do JSON ou default
    n_gens_current = exp_config.get("N_GENERATIONS_AG", GLOBAL_N_GENERATIONS_AG) # Pega do JSON ou default
    
    # Calcular MUTATION_STRENGTH_AG dinamicamente se não estiver no JSON, senão usa o do JSON
    mut_strength_from_json = exp_config.get("MUTATION_STRENGTH_AG")
    if mut_strength_from_json is not None:
        mut_strength_current = mut_strength_from_json
    else:
        mut_strength_current = max(1, math.ceil(0.01 * l0_size_current))
    
    optimization_dir_for_l0 = f"ag_optimization_results_L0_{l0_size_current}"
    os.makedirs(optimization_dir_for_l0, exist_ok=True) # Garante que o diretório existe
    
    crossover_rate_current = exp_config.get("CROSSOVER_RATE_AG", DEFAULT_CROSSOVER_RATE_AG)
    mutation_rate_current = exp_config.get("MUTATION_RATE_AG", DEFAULT_MUTATION_RATE_AG)
    elitism_rate_current = exp_config.get("ELITISM_RATE_AG", DEFAULT_ELITISM_RATE_AG)
    tournament_size_current = exp_config.get("TOURNAMENT_SIZE_AG", DEFAULT_TOURNAMENT_SIZE_AG)
    log_detailed_current = exp_config.get("LOG_DETAILED_FITNESS_AG", DEFAULT_LOG_DETAILED_FITNESS)

    print(f"Parâmetros para L0={l0_size_current}: Pop={pop_size_current}, Gens={n_gens_current}, MutStr={mut_strength_current}")
    print(f"  Crossover={crossover_rate_current}, MutRate={mutation_rate_current}, Elitism={elitism_rate_current}, TournSize={tournament_size_current}")
    print(f"Diretório de saída para este L0: {os.path.abspath(optimization_dir_for_l0)}")

    # --- SALVAR PARÂMETROS DO EXPERIMENTO ATUAL (para este L0_SIZE) ---
    experiment_params_to_save = {
        "L0_SIZE_TO_OPTIMIZE": l0_size_current,
        "POPULATION_SIZE_AG": pop_size_current,
        "N_GENERATIONS_AG": n_gens_current,
        "MUTATION_STRENGTH_AG": mut_strength_current,
        "CROSSOVER_RATE_AG": crossover_rate_current,
        "MUTATION_RATE_AG": mutation_rate_current,
        "ELITISM_RATE_AG": elitism_rate_current,
        "TOURNAMENT_SIZE_AG": tournament_size_current,
        "LOG_DETAILED_FITNESS_AG": log_detailed_current,
        "CLASSIFIER_CONFIG_AG": CLASSIFIER_SETUP_AG, # Salva a config do classificador
        "GLOBAL_EMBEDDER_CONFIG_AG": GLOBAL_EMBEDDER_SETUP_AG, # Salva a config do embedder
        "DATA_FILE_PATH": DATA_FILE_PATH,
        "TEXT_COLUMN_NAME": TEXT_COLUMN_NAME,
        "LABEL_COLUMN_NAME": LABEL_COLUMN_NAME,
        "MIN_SAMPLES_PER_CLASS_PREPROC": MIN_SAMPLES_PER_CLASS_PREPROC,
        "RARE_LABEL_GROUP": RARE_LABEL_GROUP,
        "TEST_SET_FRACTION": TEST_SET_FRACTION,
        "RANDOM_SEED_SPLIT": RANDOM_SEED_SPLIT
    }
    params_file_path = os.path.join(optimization_dir_for_l0, "experiment_params.json")
    try:
        with open(params_file_path, 'w') as f_params:
            json.dump(experiment_params_to_save, f_params, indent=4)
        print(f"  Parâmetros do experimento salvos em: {params_file_path}")
    except Exception as e_save_params:
        print(f"  AVISO: Não foi possível salvar os parâmetros do experimento: {e_save_params}")
    # --- FIM SALVAR PARÂMETROS ---

    if df_train_pool_main.empty or l0_size_current > len(df_train_pool_main): 
        print(f"AVISO: L0_SIZE_TO_OPTIMIZE ({l0_size_current}) inválido para o pool de treino (tamanho: {len(df_train_pool_main)}). Pulando este L0.")
        continue

    optimization_scenarios = [
        {'goal': 'maximize', 'metric': 'accuracy_on_full', 'seed_offset': 0},
        {'goal': 'minimize', 'metric': 'accuracy_on_full', 'seed_offset': 1000},
        {'goal': 'maximize', 'metric': 'f1_macro_on_full', 'seed_offset': 2000},
        {'goal': 'minimize', 'metric': 'f1_macro_on_full', 'seed_offset': 3000},
    ]

    for scenario in optimization_scenarios:
        execute_single_ag_optimization(
            df_l0_sampling_pool=df_train_pool_main,
            df_evaluation_set=df_test_eval_main,
            all_possible_labels_list=all_possible_labels_main,
            global_embedder_instance=embedder_instance_main_ag,
            l0_size_optim=l0_size_current,
            population_size_to_use=pop_size_current,
            n_generations_to_use=n_gens_current, 
            mutation_strength_to_use=mut_strength_current,
            crossover_rate_ag=crossover_rate_current,
            mutation_rate_ag=mutation_rate_current,
            elitism_rate_ag=elitism_rate_current,
            tournament_size_ag=tournament_size_current,
            log_detailed_fitness_ag=log_detailed_current,
            optimization_goal_scenario=scenario['goal'],
            fitness_metric_name_scenario=scenario['metric'],
            base_output_dir_experiment=optimization_dir_for_l0,
            seed_offset_val=scenario['seed_offset'],
            force_restart_this_ag_run=FORCE_RESTART_ALL_OPTIMIZATIONS 
        )
    print(f"\n{'='*20} EXPERIMENTO {exp_idx + 1} PARA L0={l0_size_current} CONCLUÍDO/PULADO {'='*20}")

print("\n\nTODOS OS EXPERIMENTOS DA CONFIGURAÇÃO JSON FORAM PROCESSADOS.")

Módulos da biblioteca activetextclassification importados.
Configurações globais fixas definidas.
>>> Etapa 3: Preparação Inicial dos Dados (Divisão Treino/Teste) <<<
--- Iniciando Carga, Pré-processamento e Divisão de Dados ---
Carregando conjuntos de dados divididos e labels de '..\data_splits_cache'...
Dados carregados com sucesso.

Pool de Treino/Otimização (D_train_opt) principal: 175255 amostras
Conjunto de Teste T principal: 75110 amostras
Total de labels únicos para F1-score: 622

>>> Etapa 4: Carregar Configurações e Executar Otimizações AG <<<
Configurações de experimento carregadas de: experiments_config.json


==================== EXECUTANDO EXPERIMENTO 1 / 12 ====================
Parâmetros para L0=10: Pop=50, Gens=100, MutStr=1
  Crossover=0.8, MutRate=0.1, Elitism=0.1, TournSize=3
Diretório de saída para este L0: d:\Nuvem\ghdaru\OneDrive\030_DOUTORADO\120_TESE\130_TESEGIT\activetextclassification\examples\ag_optimization_results_L0_10
  Parâmetros do experimento salvos e

Criando População Inicial:   0%|          | 0/50 [00:00<?, ?it/s]

AG (Melhor: -inf):   0%|          | 0/100 [00:00<?, ?it/s]

Fitness Gen 1:   0%|          | 0/50 [00:00<?, ?it/s]

Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 10 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 10 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 10 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2

...

Fitness Gen 19:   0%|          | 0/50 [00:00<?, ?it/s]

Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 100 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 100 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 100 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1

Fitness Gen 20:   0%|          | 0/50 [00:00<?, ?it/s]

Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 100 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 100 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1, 2]}
Model Factory (PVBin): Convertido ngram_range [1, 2] para tupla (1, 2)
Fitting ProductVectorizerClassifier com 100 amostras...
Fit concluído.
Model Factory: Criando tipo 'PVBin' com params: {'method': 'binary', 'query': 'binary', 'norm': None, 'query_norm': None, 'ngram_range': [1

KeyboardInterrupt: 